# 4주차 — SELECT ② WHERE 조건
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. 비교 연산과 날짜

In [ ]:
# 여성 환자
q("""
SELECT patient_id, sex, birth_date FROM patient WHERE sex = 'F' LIMIT 8
""")

In [ ]:
# 1950년 이전 출생
q("""
SELECT patient_id, birth_date FROM patient
WHERE birth_date < '1950-01-01' ORDER BY birth_date
""")

## 2. AND / OR / IN / BETWEEN

In [ ]:
# 65세 이상 여성 (2026년 기준 연 나이)
q("""
SELECT patient_id, sex, birth_date,
       2026 - CAST(substr(birth_date,1,4) AS INT) AS age
FROM patient
WHERE sex = 'F'
  AND 2026 - CAST(substr(birth_date,1,4) AS INT) >= 65
""")

In [ ]:
# 응급 또는 입원 (IN)
q("""
SELECT visit_id, visit_date, visit_type FROM visit
WHERE visit_type IN ('응급','입원') ORDER BY visit_date DESC LIMIT 8
""")

In [ ]:
# 2024년 내원 (BETWEEN)
q("""
SELECT visit_id, visit_date, department FROM visit
WHERE visit_date BETWEEN '2024-01-01' AND '2024-12-31'
""")

## 3. LIKE — 코드 패턴

In [ ]:
# 순환기계(I코드) 진단 전체
q("""
SELECT dx_id, kcd_code, dx_name FROM diagnosis
WHERE kcd_code LIKE 'I%' ORDER BY kcd_code LIMIT 10
""")

In [ ]:
# 당뇨(E11) 패밀리
q("""
SELECT DISTINCT kcd_code, dx_name FROM diagnosis WHERE kcd_code LIKE 'E11%'
""")

## 4. IS NULL

In [ ]:
# 퇴원일이 없는 내원 (외래·응급 귀가)
q("""
SELECT visit_id, visit_type, visit_date, discharge_date
FROM visit WHERE discharge_date IS NULL LIMIT 8
""")

⚠️ `discharge_date = NULL` 로 바꿔 실행해 보세요 — **아무 행도 안 나옵니다.** NULL은 `IS NULL`로만.

## 연습문제

In [ ]:
# [연습 Q1] 남성이면서 O+ 혈액형인 환자는?
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 2023년 이후 신경과 내원을 최신순으로
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 호흡기계(J코드) 진단을 모두 찾아라
# 힌트: LIKE 'J%'
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 투약일수가 30~90일인 처방은?
# 힌트: BETWEEN
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q5] 정상 상한(ref_high)이 기록되지 않은 검사가 있는가?
# 힌트: IS NULL
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 남성 O+**
```sql
SELECT * FROM patient WHERE sex = 'M' AND blood_type = 'O+';
```

**A2. 2023년 이후 신경과**
```sql
SELECT * FROM visit WHERE department = '신경과' AND visit_date >= '2023-01-01' ORDER BY visit_date DESC;
```

**A3. J코드 진단**
```sql
SELECT * FROM diagnosis WHERE kcd_code LIKE 'J%';
```

**A4. 30~90일 처방**
```sql
SELECT * FROM prescription WHERE duration_days BETWEEN 30 AND 90;
```

**A5. ref_high 없는 검사**
```sql
SELECT * FROM lab WHERE ref_high IS NULL;  -- 이 데이터셋에는 0행 — '없다'도 답이다
```